# Notebook 05 — Cross-Sample Comparison

**Goal**: Compare IRT parameters and measurement properties across Sample 1 (pretest, N~1035) and Sample 2 (validation, N~800). DIF analysis, parameter stability, measurement invariance.

In [1]:
import os, warnings
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
import girth

warnings.filterwarnings('ignore')

_MARKER = "data/processed/irt_pipeline/sample1_pretest.csv"
PROJECT_ROOT = os.path.abspath(os.getcwd())
while PROJECT_ROOT:
    if os.path.isfile(os.path.join(PROJECT_ROOT, _MARKER)):
        break
    _parent = os.path.dirname(PROJECT_ROOT)
    if _parent == PROJECT_ROOT:
        PROJECT_ROOT = os.path.abspath(os.getcwd())
        break
    PROJECT_ROOT = _parent

DATA_DIR = os.path.join(PROJECT_ROOT, "data/processed/irt_pipeline")
IRT_DIR = os.path.join(PROJECT_ROOT, "data/processed/results/03_irt_fitting")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "data/processed/results/05_cross_sample")
os.makedirs(RESULTS_DIR, exist_ok=True)

# Load data
s1_auth = pd.read_csv(os.path.join(DATA_DIR, "sample1_authors_only.csv")).apply(pd.to_numeric, errors='coerce')
s2_auth = pd.read_csv(os.path.join(DATA_DIR, "sample2_authors_only.csv")).apply(pd.to_numeric, errors='coerce')
meta = pd.read_csv(os.path.join(DATA_DIR, "item_metadata.csv"))

p1 = np.load(os.path.join(IRT_DIR, 'irt_2pl_params_sample1.npz'))
p2 = np.load(os.path.join(IRT_DIR, 'irt_2pl_params_sample2.npz'))
a1, b1 = p1['a'], p1['b']
a2, b2 = p2['a'], p2['b']

author_cols = s1_auth.columns.tolist()
X1 = s1_auth.dropna().values.astype(int)
X2 = s2_auth.dropna().values.astype(int)

theta1 = girth.ability_eap(X1.T, b1, a1)
theta2 = girth.ability_eap(X2.T, b2, a2)

print(f"Sample 1: {X1.shape}, Sample 2: {X2.shape}")
print(f"Parameters: {len(a1)} items")

Sample 1: (449, 101), Sample 2: (798, 101)
Parameters: 101 items


---
## 5a. Item Parameter Comparison

In [2]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Difficulty scatter
axes[0].scatter(b1, b2, alpha=0.6, s=25)
lim_b = [min(b1.min(), b2.min()) - 0.5, max(b1.max(), b2.max()) + 0.5]
axes[0].plot(lim_b, lim_b, 'r--', alpha=0.5)
axes[0].set_xlabel('Sample 1 — b (difficulty)')
axes[0].set_ylabel('Sample 2 — b (difficulty)')
axes[0].set_title('Difficulty Parameter Comparison')
r_b, p_b = stats.pearsonr(b1, b2)
axes[0].annotate(f'r = {r_b:.3f}\np = {p_b:.2e}', xy=(0.05, 0.88),
                 xycoords='axes fraction', fontsize=11)

# Discrimination scatter
axes[1].scatter(a1, a2, alpha=0.6, s=25)
lim_a = [0, max(a1.max(), a2.max()) + 0.5]
axes[1].plot(lim_a, lim_a, 'r--', alpha=0.5)
axes[1].set_xlabel('Sample 1 — a (discrimination)')
axes[1].set_ylabel('Sample 2 — a (discrimination)')
axes[1].set_title('Discrimination Parameter Comparison')
r_a, p_a = stats.pearsonr(a1, a2)
axes[1].annotate(f'r = {r_a:.3f}\np = {p_a:.2e}', xy=(0.05, 0.88),
                 xycoords='axes fraction', fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'parameter_comparison.png'), dpi=150)
plt.show()

# MAD and large shifts
delta_b = np.abs(b1 - b2)
delta_a = np.abs(a1 - a2)
print(f"\nDifficulty (b):")
print(f"  Pearson r = {r_b:.4f}, p = {p_b:.2e}")
print(f"  Mean absolute difference: {delta_b.mean():.3f}")
print(f"  Items with |delta_b| > 0.50: {(delta_b > 0.50).sum()}")

print(f"\nDiscrimination (a):")
print(f"  Pearson r = {r_a:.4f}, p = {p_a:.2e}")
print(f"  Mean absolute difference: {delta_a.mean():.3f}")
print(f"  Items with |delta_a| > 0.30: {(delta_a > 0.30).sum()}")


Difficulty (b):
  Pearson r = 0.2996, p = 2.34e-03
  Mean absolute difference: 1.824
  Items with |delta_b| > 0.50: 48

Discrimination (a):
  Pearson r = 0.5548, p = 1.75e-09
  Mean absolute difference: 0.498
  Items with |delta_a| > 0.30: 51


---
## 5b. Differential Item Functioning (DIF)

In [3]:
def lord_chi_square_dif(a1, b1, a2, b2, se_scale=0.1):
    """Lord's chi-square DIF test comparing 2PL parameters across groups.
    Uses approximate SE based on parameter scale.
    """
    n_items = len(a1)
    results = []
    
    for j in range(n_items):
        # Parameter differences
        d_a = a1[j] - a2[j]
        d_b = b1[j] - b2[j]
        
        # Approximate variance-covariance (diagonal)
        se_a = se_scale * max(abs(a1[j]), abs(a2[j]), 0.5)
        se_b = se_scale * max(abs(b1[j]), abs(b2[j]), 0.5)
        
        chi2 = (d_a / se_a)**2 + (d_b / se_b)**2
        p_val = 1 - stats.chi2.cdf(chi2, df=2)
        
        results.append({
            'item': author_cols[j],
            'delta_a': d_a,
            'delta_b': d_b,
            'lord_chi2': chi2,
            'lord_p': p_val
        })
    
    return pd.DataFrame(results)

lord_dif = lord_chi_square_dif(a1, b1, a2, b2)
bonf = 0.05 / len(a1)
lord_dif['significant_bonf'] = lord_dif['lord_p'] < bonf

print("="*60)
print("Lord's Chi-Square DIF Test")
print("="*60)
print(f"  Items tested: {len(lord_dif)}")
print(f"  Bonferroni threshold: p < {bonf:.4f}")
print(f"  Significant DIF items: {lord_dif['significant_bonf'].sum()}")

if lord_dif['significant_bonf'].sum() > 0:
    sig_items = lord_dif[lord_dif['significant_bonf']].sort_values('lord_chi2', ascending=False)
    print(f"\n  Top DIF items:")
    print(sig_items[['item', 'delta_a', 'delta_b', 'lord_chi2', 'lord_p']].head(10).to_string(index=False))

Lord's Chi-Square DIF Test
  Items tested: 101
  Bonferroni threshold: p < 0.0005
  Significant DIF items: 55

  Top DIF items:
                 item   delta_a   delta_b  lord_chi2  lord_p
    Margaret Mitchell -1.255046 -1.298479 378.571206     0.0
          Jules Verne  0.763187 -9.304416 275.077235     0.0
       Victor Pelevin -0.224572 -2.549829 235.123093     0.0
    Michel Houlleback  0.207713  0.762646 233.344464     0.0
         Bernard Shaw  2.052303 -7.503135 209.725678     0.0
         Kir Bulychev  0.726020 -1.287367 208.639825     0.0
Gregory David Roberts -0.989965  4.383429 185.250233     0.0
            Dan Bruwn  1.001560 -7.021977 180.829005     0.0
    Catherine Stokett -1.899015  6.315996 176.047982     0.0
           Harper Lee  1.390956 -6.572076 171.530148     0.0


In [4]:
def raju_area_dif(a1, b1, a2, b2):
    """Raju's unsigned area between two ICCs."""
    theta = np.linspace(-4, 4, 1000)
    n_items = len(a1)
    results = []
    
    for j in range(n_items):
        p1 = 1.0 / (1.0 + np.exp(-a1[j] * (theta - b1[j])))
        p2 = 1.0 / (1.0 + np.exp(-a2[j] * (theta - b2[j])))
        area = np.trapezoid(np.abs(p1 - p2), theta)
        results.append({'item': author_cols[j], 'raju_area': area})
    
    return pd.DataFrame(results)

raju = raju_area_dif(a1, b1, a2, b2)
print("\n" + "="*60)
print("Raju's Unsigned Area DIF")
print("="*60)
print(f"  Mean area: {raju['raju_area'].mean():.4f}")
print(f"  SD: {raju['raju_area'].std():.4f}")
print(f"  Items with area > 0.40 (large DIF): {(raju['raju_area'] > 0.40).sum()}")
print(f"\n  Top-10 by area:")
print(raju.nlargest(10, 'raju_area').to_string(index=False))


Raju's Unsigned Area DIF
  Mean area: 1.2539
  SD: 1.5548
  Items with area > 0.40 (large DIF): 52

  Top-10 by area:
            item  raju_area
     Jules Verne   6.691553
    Bernard Shaw   5.332457
       Dan Bruwn   4.593155
    Dmitry Bykov   4.470631
     Neil Gaiman   4.424584
 Zakhar Prilepin   4.157827
      Harper Lee   4.146137
Dmutry Gluhovsky   4.054481
        Ayn Rand   3.951058
    Daniel Keyes   3.813144


In [5]:
def mantel_haenszel_dif(s1_auth, s2_auth, author_cols, n_strata=5):
    """Mantel-Haenszel DIF test stratified by total score."""
    d1 = s1_auth.dropna()
    d2 = s2_auth.dropna()
    
    total1 = d1.sum(axis=1)
    total2 = d2.sum(axis=1)
    
    # Create score strata using combined quantiles
    all_totals = np.concatenate([total1.values, total2.values])
    try:
        strata_edges = np.percentile(all_totals, np.linspace(0, 100, n_strata + 1))
        strata_edges = np.unique(strata_edges)
    except Exception:
        strata_edges = np.linspace(all_totals.min(), all_totals.max(), n_strata + 1)
    
    strata1 = np.digitize(total1, strata_edges[1:-1])
    strata2 = np.digitize(total2, strata_edges[1:-1])
    
    results = []
    for j, col in enumerate(author_cols):
        x1 = d1[col].values
        x2 = d2[col].values
        
        mh_num = 0.0
        mh_den = 0.0
        
        for s in np.unique(np.concatenate([strata1, strata2])):
            m1 = strata1 == s
            m2 = strata2 == s
            
            n1s = m1.sum()
            n2s = m2.sum()
            if n1s < 2 or n2s < 2:
                continue
            
            a = x1[m1].sum()  # endorsed in group 1
            b = n1s - a       # not endorsed in group 1
            c = x2[m2].sum()  # endorsed in group 2
            d = n2s - c       # not endorsed in group 2
            
            n_s = n1s + n2s
            if n_s > 0:
                mh_num += a * d / n_s
                mh_den += b * c / n_s
        
        if mh_den > 0:
            alpha_mh = mh_num / mh_den
            delta_mh = -2.35 * np.log(alpha_mh) if alpha_mh > 0 else 0
        else:
            alpha_mh = np.inf
            delta_mh = 0
        
        # ETS classification
        abs_delta = abs(delta_mh)
        if abs_delta < 1.0:
            ets_class = 'A'
        elif abs_delta < 1.5:
            ets_class = 'B'
        else:
            ets_class = 'C'
        
        results.append({
            'item': col,
            'alpha_MH': alpha_mh,
            'delta_MH': delta_mh,
            'abs_delta_MH': abs_delta,
            'ETS_class': ets_class
        })
    
    return pd.DataFrame(results)

mh = mantel_haenszel_dif(s1_auth, s2_auth, author_cols)
print("\n" + "="*60)
print("Mantel-Haenszel DIF")
print("="*60)
ets_counts = mh['ETS_class'].value_counts().sort_index()
print(f"  ETS classification:")
for cls, cnt in ets_counts.items():
    label = {'A': 'Negligible', 'B': 'Moderate', 'C': 'Large'}[cls]
    print(f"    {cls} ({label}): {cnt} items")

if (mh['ETS_class'] == 'C').sum() > 0:
    print(f"\n  Items with large DIF (C):")
    c_items = mh[mh['ETS_class'] == 'C'].sort_values('abs_delta_MH', ascending=False)
    print(c_items[['item', 'delta_MH', 'ETS_class']].to_string(index=False))

mh.to_csv(os.path.join(RESULTS_DIR, 'dif_mantel_haenszel.csv'), index=False)


Mantel-Haenszel DIF
  ETS classification:
    A (Negligible): 20 items
    B (Moderate): 15 items
    C (Large): 66 items

  Items with large DIF (C):
                   item   delta_MH ETS_class
            Jules Verne -18.390069         C
           Bernard Shaw -15.685535         C
           Dmitry Bykov -13.437891         C
  Gregory David Roberts  13.102651         C
              Dan Bruwn -10.847650         C
         Victor Pelevin -10.721094         C
        Zakhar Prilepin -10.576476         C
               Ayn Rand  -9.951973         C
            Neil Gaiman  -9.939916         C
       Dmutry Gluhovsky  -9.936640         C
           Yury  Olesha   9.882735         C
  Reshad Nuri Gyuntekin   9.268029         C
      Catherine Stokett   9.248652         C
             Harper Lee  -9.162305         C
            Isaac Babel   9.048447         C
             Alan Milne  -8.467344         C
            Dina Rubina  -8.060343         C
   Ethel Lilian Voynich  -8.042329    

---
## 5c. Test-Level Comparison

In [6]:
# TIF comparison
theta_range = np.linspace(-4, 4, 200)

tif1 = np.zeros(len(theta_range))
tif2 = np.zeros(len(theta_range))
for j in range(len(a1)):
    P1 = 1.0 / (1.0 + np.exp(-a1[j] * (theta_range - b1[j])))
    P2 = 1.0 / (1.0 + np.exp(-a2[j] * (theta_range - b2[j])))
    tif1 += a1[j]**2 * P1 * (1 - P1)
    tif2 += a2[j]**2 * P2 * (1 - P2)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(theta_range, tif1, 'b-', linewidth=2, label='Sample 1')
ax.plot(theta_range, tif2, 'r-', linewidth=2, label='Sample 2')
ax.fill_between(theta_range, tif1, tif2, alpha=0.1, color='gray')
ax.set_xlabel('θ', fontsize=12)
ax.set_ylabel('Test Information', fontsize=12)
ax.set_title('Test Information Function Comparison', fontsize=14)
ax.legend(fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'tif_comparison.png'), dpi=150)
plt.show()

# Theta distribution comparison
t_stat, t_p = stats.ttest_ind(theta1, theta2, equal_var=False)
cohens_d = (theta1.mean() - theta2.mean()) / np.sqrt((theta1.var() + theta2.var()) / 2)
print(f"\nTheta Distribution Comparison:")
print(f"  Sample 1: M = {theta1.mean():.3f}, SD = {theta1.std():.3f}")
print(f"  Sample 2: M = {theta2.mean():.3f}, SD = {theta2.std():.3f}")
print(f"  Welch's t = {t_stat:.3f}, p = {t_p:.4f}")
print(f"  Cohen's d = {cohens_d:.3f}")

# Endorsement rate comparison
endorse1 = s1_auth.mean()
endorse2 = s2_auth.mean()
t_e, p_e = stats.ttest_rel(endorse1, endorse2)
print(f"\nEndorsement Rate Comparison (paired t on items):")
print(f"  S1 mean: {endorse1.mean():.3f}, S2 mean: {endorse2.mean():.3f}")
print(f"  t({len(endorse1)-1}) = {t_e:.3f}, p = {p_e:.4f}")


Theta Distribution Comparison:
  Sample 1: M = 0.029, SD = 1.019
  Sample 2: M = 0.000, SD = 0.976
  Welch's t = 0.482, p = 0.6298
  Cohen's d = 0.029

Endorsement Rate Comparison (paired t on items):
  S1 mean: 0.513, S2 mean: 0.416
  t(100) = 3.602, p = 0.0005


---
## 5d. Measurement Invariance Summary

In [7]:
# Combine DIF results
dif_summary = lord_dif[['item', 'delta_a', 'delta_b', 'lord_chi2', 'lord_p', 'significant_bonf']].copy()
dif_summary = dif_summary.merge(raju[['item', 'raju_area']], on='item')
dif_summary = dif_summary.merge(mh[['item', 'delta_MH', 'ETS_class']], on='item')

dif_summary.to_csv(os.path.join(RESULTS_DIR, 'dif_combined.csv'), index=False)

# Invariance summary
print("="*60)
print("MEASUREMENT INVARIANCE SUMMARY")
print("="*60)

print(f"\nParameter correlations across samples:")
print(f"  Difficulty (b): r = {r_b:.4f}, p = {p_b:.2e}")
print(f"  Discrimination (a): r = {r_a:.4f}, p = {p_a:.2e}")

print(f"\nDIF summary:")
print(f"  Lord's chi-square: {lord_dif['significant_bonf'].sum()} / {len(lord_dif)} significant (Bonferroni)")
print(f"  Raju area > 0.40: {(raju['raju_area'] > 0.40).sum()} / {len(raju)}")
print(f"  MH ETS class C: {(mh['ETS_class'] == 'C').sum()} / {len(mh)}")

n_any_dif = ((lord_dif['significant_bonf']) | 
             (raju['raju_area'].values > 0.40) | 
             (mh['ETS_class'].values == 'C')).sum()
print(f"  Items flagged by ANY method: {n_any_dif} / {len(lord_dif)}")

pct_invariant = 100 * (1 - n_any_dif / len(lord_dif))
print(f"  Invariant items: {pct_invariant:.1f}%")

if pct_invariant >= 80:
    print(f"\n  CONCLUSION: Good cross-sample stability ({pct_invariant:.0f}% invariant)")
elif pct_invariant >= 60:
    print(f"\n  CONCLUSION: Moderate cross-sample stability ({pct_invariant:.0f}% invariant)")
else:
    print(f"\n  CONCLUSION: Weak cross-sample stability ({pct_invariant:.0f}% invariant)")

print(f"\nAll results saved to: {RESULTS_DIR}")
for f in sorted(os.listdir(RESULTS_DIR)):
    print(f"  {f}")

MEASUREMENT INVARIANCE SUMMARY

Parameter correlations across samples:
  Difficulty (b): r = 0.2996, p = 2.34e-03
  Discrimination (a): r = 0.5548, p = 1.75e-09

DIF summary:
  Lord's chi-square: 55 / 101 significant (Bonferroni)
  Raju area > 0.40: 52 / 101
  MH ETS class C: 66 / 101
  Items flagged by ANY method: 86 / 101
  Invariant items: 14.9%

  CONCLUSION: Weak cross-sample stability (15% invariant)

All results saved to: /home/polina/Documents/Cursor_Projects/Russian Author Recognition Test Cursor/data/processed/results/05_cross_sample
  dif_combined.csv
  dif_mantel_haenszel.csv
  parameter_comparison.png
  tif_comparison.png
